# Road Inspection — train the India road-damage model on Kaggle's free GPU

**Before running**
1. *Settings → Accelerator*: **GPU T4 x2** (or P100). *Settings → Internet*: **On**
   (needs a phone-verified Kaggle account).
2. *Add Input*: your dataset made from `road-inspection-india.zip`.
3. Run it as a background job: **Save Version → Save & Run All (Commit)**.
   The commit keeps running after you close the browser (12 h session limit) and
   saves everything in `/kaggle/working` as the notebook's output.

When it finishes, open the version's **Output** tab and download `india_model.zip`.

**Recipe (India only).** YOLO11m at 768 px on RDD2022 India. The India *test* split is
folded into training (`--fold-test`, ~23 % more labelled damage); India *val* still
picks the best epoch and tunes per-class thresholds (F2, recall first, so minor
damage is reported). The A/B report compares against the current production model.

**How long it trains.** `HOURS` is the training budget. Ultralytics fits the
cosine LR schedule to it (~150 epochs for yolo11m on a T4), so early stopping is off
(`patience=0`) and the LR always decays fully. `best.pt` keeps the best validation epoch.

**If a session dies mid-training.** `last.pt` is written every epoch to
`/kaggle/working/runs/<run>/weights/`. Add the failed version's output as an input
(*Add Input → Your Work → this notebook*) and run again. The training cell finds
that `last.pt` and resumes with the optimizer, LR position, EMA and best score
intact, instead of starting over.

In [ ]:
# Settings — "best" = yolo11m @ 768 px (the run budget fits ~150 epochs on a T4); "laptop" = yolo11s
import time
NOTEBOOK_START = time.time()
PROFILE = "best"            # yolo11m @ 768 px
BATCH = 16                  # T4 has 15 GB; Ultralytics halves it automatically on CUDA OOM
HOURS = 10.5                # training budget (early stopping is off; the budget decides when to stop)
SESSION_HOURS = 12.0        # Kaggle hard limit for one session
RESERVE_HOURS = 0.75        # kept free for final validation, A/B evaluation and zipping
RESUME = True               # continue from a previous version's last.pt if one is attached as input
RUN_NAME = f"india_{PROFILE}"

In [ ]:
import subprocess, torch
print("CUDA:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU — enable it in Settings")
!pip install -q "ultralytics==8.4.216" pyyaml

In [ ]:
# Find the uploaded dataset (Kaggle unzips it somewhere under /kaggle/input)
from pathlib import Path
roads = sorted(p for p in Path("/kaggle/input").rglob("road_v2") if p.is_dir())
if not roads:  # dataset kept as an archive: unpack it into the working dir
    import zipfile
    archives = sorted(Path("/kaggle/input").rglob("*.zip"))
    assert archives, "road_v2 not found — did you add the dataset as input?"
    for a in archives:
        zipfile.ZipFile(a).extractall("/kaggle/working/bundle")
    roads = sorted(p for p in Path("/kaggle/working/bundle").rglob("road_v2") if p.is_dir())
assert roads, "road_v2 not found in the dataset"
ROAD = roads[0]
BUNDLE = ROAD.parent
SCRIPTS = BUNDLE / "scripts" / "training"
BASELINE = BUNDLE / "artifacts" / "ai_models" / "best.pt"
WORK = Path("/kaggle/working")
print("data:", ROAD, "| scripts:", SCRIPTS.exists(), "| baseline model:", BASELINE.exists())
for split in ("train", "val", "test"):
    print(split, len(list((ROAD / split / "images").iterdir())), "images")

In [ ]:
# 1. Build the balanced, de-duplicated training set (India train + test; val stays held out)
!python {SCRIPTS}/build_dataset.py --road {ROAD} --extra /kaggle/working/no_extra --fold-test --out {WORK}/build/india_v3

In [ ]:
# 2. Train (AMP on: it is stable on Kaggle GPUs; the collapse guard stops the run if anything goes NaN)
import shutil, time
used_h = (time.time() - NOTEBOOK_START) / 3600
budget = round(min(HOURS, SESSION_HOURS - used_h - RESERVE_HOURS), 2)
assert budget > 0.25, f"only {budget} h left in this session — start a fresh one"
print(f"setup took {used_h:.2f} h -> training budget {budget} h")

RUN_DIR = WORK / "runs" / RUN_NAME
def resumable_epoch(last):
    """Epochs done in an interrupted run, or None (finished runs drop the optimizer from last.pt)."""
    try:
        ck = torch.load(last, map_location="cpu", weights_only=False)
    except Exception as e:
        print("unreadable checkpoint, ignored:", last, e)
        return None
    return ck["epoch"] + 1 if ck.get("epoch", -1) >= 0 and ck.get("optimizer") is not None else None

candidates = []
if RESUME:
    for last in Path("/kaggle/input").rglob(f"runs/{RUN_NAME}/weights/last.pt"):
        done = resumable_epoch(last)
        print(last, "->", f"interrupted after epoch {done}" if done else "finished, not resumable")
        if done:
            candidates.append((done, last))
if candidates and not RUN_DIR.exists():
    done, last = max(candidates)
    # Kaggle inputs are read-only: copy the old run to the same path it was trained at,
    # so Ultralytics keeps writing into it (resume uses the save_dir stored in last.pt).
    shutil.copytree(last.parents[1], RUN_DIR)
    print(f"resuming {last} after epoch {done}")

if (RUN_DIR / "weights" / "last.pt").exists() and resumable_epoch(RUN_DIR / "weights" / "last.pt"):
    !python {SCRIPTS}/train.py --data {WORK}/build/india_v3/data.yaml --resume {RUN_DIR}/weights/last.pt --device 0 --workers 4 --batch {BATCH} --hours {budget}
else:
    !python {SCRIPTS}/train.py --data {WORK}/build/india_v3/data.yaml --profile {PROFILE} --batch {BATCH} --device 0 --workers 4 --amp --hours {budget} --project {WORK}/runs --name {RUN_NAME}

In [ ]:
# 3. Evaluate against the current production model on India val (report only — promotion happens on your PC)
# Ultralytics renames the folder (india_best2...) if it already exists, so take the newest run.
runs = sorted((WORK / "runs").glob(f"{RUN_NAME}*/weights/best.pt"), key=lambda f: f.stat().st_mtime)
assert runs, "training did not produce best.pt — check the training cell output"
BEST = runs[-1]
RUN_DIR = BEST.parents[1]
print("run:", RUN_DIR)
baseline_arg = f"--baseline {BASELINE}" if BASELINE.exists() else f"--baseline {BEST}"
!python {SCRIPTS}/evaluate.py --candidate {BEST} {baseline_arg} --data {WORK}/build/india_v3/data.yaml --device 0 --beta 2 || true

In [ ]:
# 4. Pack the results for download (best.pt + metrics, curves, evaluation report).
# last.pt stays in /kaggle/working/runs (part of the version output) for resuming; it is left out of the zip.
import shutil, zipfile
run = RUN_DIR
out = WORK / "india_model.zip"
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in run.rglob("*"):
        if f.is_file() and f.name != "last.pt":
            zf.write(f, f"{RUN_NAME}/{f.relative_to(run)}")
print(out, round(out.stat().st_size / 1e6, 1), "MB")
print((run / "evaluation" / "evaluation.md").read_text() if (run / "evaluation" / "evaluation.md").exists() else "no evaluation report")
for tmp in (WORK / "build", WORK / "bundle"):
    shutil.rmtree(tmp, ignore_errors=True)  # rebuilt from the input dataset on every run

## Back on your PC (project root)

1. Unzip `india_model.zip` into `artifacts/runs/` (you get `artifacts/runs/india_best/...`).
2. Re-run the gate locally and promote the model into the app:

```
python scripts/training/evaluate.py --candidate artifacts/runs/india_best/weights/best.pt --device cpu --promote
```

3. Restart the app (`start.bat`). It now uses the new model and its per-class thresholds.